# 04 — System-1: ModernBERT candidate classification and calibration

Train a real binary classifier using labeled SQL candidates. Calibration and test questions must be disjoint from training questions. Candidate execution labels are computed against gold results; gold SQL is never included in classifier input. Export classifier weights, tokenizer and temperature. This is BCE+Brier calibration-aware optimization, not a claim of demonstrated research performance.


In [ ]:
import os
import sys
from pathlib import Path

# Attach a repository snapshot as a Kaggle Dataset, or set KAGGLE_REPO_DIR.
_candidates = [Path(os.environ.get("KAGGLE_REPO_DIR", ".")), Path.cwd(), Path.cwd().parent]
_candidates += list(Path("/kaggle/input").glob("*/Optimize-Mars-Sl")) if Path("/kaggle/input").exists() else []
REPO_DIR = next((p.resolve() for p in _candidates if (p / "src").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Set KAGGLE_REPO_DIR to the attached repository directory containing src/")
sys.path.insert(0, str(REPO_DIR))
from src.training.kaggle import KagglePaths, install_dependencies, environment_report
install_dependencies(REPO_DIR)  # Only runs pip when INSTALL_DEPS=1.
paths = KagglePaths.from_env()
SMOKE = paths.smoke
print({"mode": "smoke (NOT a benchmark)" if SMOKE else "train", "data": str(paths.data_dir), "device": paths.device})


In [ ]:
PHASE = 'system1'
import json
OUTPUT_DIR = paths.output(PHASE)
environment = environment_report(OUTPUT_DIR)
MAX_STEPS = int(os.environ.get("MAX_STEPS", "100"))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "512"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", "25"))
RESUME = os.environ.get("RESUME_FROM_CHECKPOINT") or None  # "latest" or attached checkpoint path
print({"output": OUTPUT_DIR, "resume_from_checkpoint": RESUME, "gpus": environment["gpus"]})


Set `DECISION_DATA`, `CALIBRATION_DATA`, `DECISION_TEST_DATA` to labeled JSONL (question, schema, sql, label); use `DB_DIR` for label construction from supplied candidate SQL. See KAGGLE.md for the candidate-data builder. System-1 may be trained independently from existing candidate artifacts on a separate legitimate account.

In [ ]:
from src.training.train_decision_model import run_decision_model_training, build_candidate_dataset, split_candidate_groups
DECISION_DATA = paths.input("DECISION_DATA", "decision_train.jsonl")
CALIBRATION_DATA = paths.input("CALIBRATION_DATA", "decision_calibration.jsonl")
DECISION_TEST_DATA = paths.input("DECISION_TEST_DATA", "decision_test.jsonl")
# Optional: construct execution labels from an independently sampled candidate artifact.
# Otherwise attach the three prebuilt labeled datasets documented above.
if not SMOKE and os.environ.get("CANDIDATE_SOURCE"):
    labeled = build_candidate_dataset(os.environ["CANDIDATE_SOURCE"],
        db_dir=paths.input("DB_DIR", "databases"), include_gold=False)
    grouped = split_candidate_groups(labeled, seed=SEED)
    destinations = [Path(OUTPUT_DIR) / f"decision_{name}.jsonl" for name in ("train", "calibration", "test")]
    for destination, rows in zip(destinations, grouped):
        with destination.open("x") as handle:
            handle.write("".join(json.dumps(row) + "\n" for row in rows))
    DECISION_DATA, CALIBRATION_DATA, DECISION_TEST_DATA = map(str, destinations)
summary = run_decision_model_training(data_path=DECISION_DATA, output_dir=OUTPUT_DIR,
    calibration_path=CALIBRATION_DATA, test_path=DECISION_TEST_DATA, dry_run=SMOKE,
    model_name_or_path=os.environ.get("DECISION_MODEL", "answerdotai/ModernBERT-large"),
    device=paths.device, max_steps=MAX_STEPS, max_length=MAX_LENGTH, save_steps=SAVE_STEPS,
    seed=SEED, resume_from_checkpoint=RESUME)
print(json.dumps(summary, indent=2))


## Artifact handoff and validation

The last cell returns artifact paths. Download `/kaggle/working/optimize-mars` after a run or save a notebook version with outputs, then attach those outputs to the downstream notebook as a Kaggle Dataset. Runtime storage is ephemeral; a local checkpoint alone does not survive session deletion. Export before the session ends. Set `RESUME_FROM_CHECKPOINT` to an attached complete checkpoint, or `latest` for the current output directory. No checkpoints are automatically deleted.

Use `PIPELINE_MODE=smoke` for offline wiring checks; this mode does not train pretrained models or measure model quality. Default `train` requires real datasets, dependencies, GPU and the input artifacts listed above. Settings target **one 16 GB T4**, sequential rollouts and short sequences; T4 x2 does not provide a shared 32 GB memory pool. Reduce `MAX_LENGTH`, rollout length or accumulation if needed. No full GPU run or Kaggle benchmark is claimed by this notebook. See `KAGGLE.md` for data contracts, independent account handoffs and ablations.
